# MT3510 Group Project — DFA Synchronization & Isomorphism

This notebook demonstrates all implemented algorithms.

In [ ]:
from dfa.core import (
    validate_dfa, apply_word, apply_word_to_set, is_synchronizing_word,
    random_dfa, non_synchronizing_dfa, dfa_to_digraph
)
from sync.monoid import is_synchronizing_monoid
from sync.images import is_synchronizing_images, shortest_reset_word_images
from sync.pair_graph import is_synchronizing_pair_graph, reset_word_pair_graph, build_pair_graph
from sync.heuristic import shorten_reset_word, compare_reset_words
from iso.isomorphism import is_isomorphic, is_weakly_isomorphic, is_semi_isomorphic
import networkx as nx
import time

## 1. The Example DFA from the Spec

States: {0, 1, 2, 3}, Alphabet: {0, 1}, Start: 3, Final: {1, 2}

Known properties:
- Synchronizing (reset words: `11` → state 0, `10` → state 1)
- Accepts words `0`, `10`, `110` via paths 3→2, 3→0→1, 3→0→0→1

In [ ]:
# Spec DFA: verify the transitions match the diagram
SPEC_DFA = (
    {0, 1, 2, 3},
    {0, 1},
    {
        (0, 0): 0, (0, 1): 0,
        (1, 0): 0, (1, 1): 0,
        (2, 0): 1, (2, 1): 0,
        (3, 0): 0, (3, 1): 2,
    },
    3,
    {1, 2},
)

validate_dfa(SPEC_DFA)
print('Spec DFA is valid')
print(f'Word [1,1] is reset word: {is_synchronizing_word(SPEC_DFA, [1, 1])}')
print(f'Word [1,0] is reset word: {is_synchronizing_word(SPEC_DFA, [1, 0])}')
print(f'Applying [0] from start state 3: ends in state {apply_word(SPEC_DFA, 3, [0])}')
print(f'Applying [1,0] from start state 3: ends in state {apply_word(SPEC_DFA, 3, [1, 0])}')

## 2. Part 1 — Synchronization

### Task (a): Transition Monoid BFS

In [ ]:
print('=== Task (a): Transition Monoid BFS ===')
print(f'Spec DFA synchronizing: {is_synchronizing_monoid(SPEC_DFA)}')

non_sync = non_synchronizing_dfa(4, 2)
print(f'Non-sync DFA synchronizing: {is_synchronizing_monoid(non_sync)}')

# Timing on a small random DFA
A_small = random_dfa(8, 2, seed=100)
t0 = time.time()
result = is_synchronizing_monoid(A_small)
t1 = time.time()
print(f'Random DFA (n=8, m=2) synchronizing: {result} (took {t1-t0:.4f}s)')

# WARNING: transition monoid grows fast!
# For n=10, m=2 the monoid can already be very large.
A_10 = random_dfa(10, 2, seed=200)
t0 = time.time()
result = is_synchronizing_monoid(A_10)
t1 = time.time()
print(f'Random DFA (n=10, m=2) synchronizing: {result} (took {t1-t0:.4f}s)')

### Task (b): Image BFS

In [ ]:
print('=== Task (b): Image BFS ===')
print(f'Spec DFA synchronizing: {is_synchronizing_images(SPEC_DFA)}')
print(f'Non-sync DFA synchronizing: {is_synchronizing_images(non_sync)}')

# Image BFS is much more scalable
for n in [10, 20, 50, 100]:
    A = random_dfa(n, 2, seed=42)
    t0 = time.time()
    result = is_synchronizing_images(A)
    t1 = time.time()
    print(f'  n={n:3d}: synchronizing={result}, time={t1-t0:.4f}s')

### Task (c): Pair Graph Method

In [ ]:
print('=== Task (c): Pair Graph Method ===')
print(f'Spec DFA synchronizing: {is_synchronizing_pair_graph(SPEC_DFA)}')
print(f'Non-sync DFA synchronizing: {is_synchronizing_pair_graph(non_sync)}')

# Pair graph has O(n^2) vertices
for n in [10, 20, 50]:
    A = random_dfa(n, 2, seed=42)
    t0 = time.time()
    result = is_synchronizing_pair_graph(A)
    t1 = time.time()
    print(f'  n={n:3d}: synchronizing={result}, time={t1-t0:.4f}s')

### Agreement of all three methods

In [ ]:
print('Checking agreement across methods on 20 random DFAs (n=6)...')
all_agree = True
for seed in range(20):
    A = random_dfa(6, 2, seed=seed)
    r1 = is_synchronizing_monoid(A)
    r2 = is_synchronizing_images(A)
    r3 = is_synchronizing_pair_graph(A)
    if not (r1 == r2 == r3):
        print(f'  DISAGREEMENT at seed {seed}: monoid={r1}, images={r2}, pair_graph={r3}')
        all_agree = False
if all_agree:
    print('All three methods agree on all 20 DFAs ✓')

### Task (d): Shortest Reset Word (Image BFS with Ancestors)

In [ ]:
print('=== Task (d): Shortest Reset Word (Image BFS) ===')
word = shortest_reset_word_images(SPEC_DFA)
print(f'Spec DFA shortest reset word: {word} (length {len(word)})')
print(f'  Verification: is_synchronizing_word = {is_synchronizing_word(SPEC_DFA, word)}')
print(f'  All states end in: {apply_word_to_set(SPEC_DFA, SPEC_DFA[0], word)}')

# Non-synchronizing DFA should return None
print(f'Non-sync DFA: {shortest_reset_word_images(non_sync)}')

# Larger example
A = random_dfa(20, 2, seed=42)
word = shortest_reset_word_images(A)
if word:
    print(f'Random (n=20) shortest reset word length: {len(word)}')
    print(f'  Černý bound: {(20-1)**2} = {19**2}')
    print(f'  Verification: {is_synchronizing_word(A, word)}')

### Task (e): Reset Word via Pair Graph

In [ ]:
print('=== Task (e): Reset Word (Pair Graph) ===')
word_pg = reset_word_pair_graph(SPEC_DFA)
print(f'Spec DFA pair graph reset word: {word_pg} (length {len(word_pg)})')
print(f'  Verification: {is_synchronizing_word(SPEC_DFA, word_pg)}')

print(f'Non-sync DFA: {reset_word_pair_graph(non_sync)}')

A = random_dfa(20, 2, seed=42)
word_pg = reset_word_pair_graph(A)
if word_pg:
    print(f'Random (n=20) pair graph reset word length: {len(word_pg)}')
    print(f'  Verification: {is_synchronizing_word(A, word_pg)}')

### Task (f): Comparison and Heuristic Shortening

In [ ]:
print('=== Task (f): Comparison & Heuristic Shortening ===')
print()

# Compare on multiple random DFAs
print(f'{"n":>4} {"Shortest":>10} {"Pair Graph":>12} {"Greedy":>8} {"Window":>8} {"Suffix":>8} {"Černý":>8}')
print('-' * 65)

for n in [5, 8, 10, 15, 20, 30]:
    A = random_dfa(n, 2, seed=42)
    w_img = shortest_reset_word_images(A)
    w_pg = reset_word_pair_graph(A)
    cerny = (n - 1) ** 2
    
    if w_img is not None and w_pg is not None:
        w_greedy = shorten_reset_word(A, w_pg, method='greedy')
        w_window = shorten_reset_word(A, w_pg, method='window')
        w_suffix = shorten_reset_word(A, w_pg, method='suffix')
        print(f'{n:4d} {len(w_img):10d} {len(w_pg):12d} {len(w_greedy):8d} {len(w_window):8d} {len(w_suffix):8d} {cerny:8d}')
    else:
        print(f'{n:4d}  Not synchronizing')

In [ ]:
# Detailed comparison on one DFA
A = random_dfa(15, 2, seed=42)
stats = compare_reset_words(A, shortest_reset_word_images(A), reset_word_pair_graph(A))
print('Detailed comparison (n=15):')
for k, v in stats.items():
    if k != 'shortened_word':
        print(f'  {k}: {v}')

### Analysis of Heuristics

**Why the pair graph word is longer:**
The pair graph method works by iteratively merging pairs of states. Each merge uses the shortest word to collapse *that specific pair*, but applying the merge word to all remaining states can re-split previously merged states. This greedy pair-by-pair approach doesn't optimise globally.

**Greedy trimming:** Tries removing each letter one at a time. Works well because many letters in the pair graph word are redundant — they were needed for a specific merge step but become unnecessary after concatenation.

**Suffix trimming:** Binary searches for the shortest prefix that is still a reset word, then applies greedy. This is effective when the tail of the concatenated word contains redundant letters.

**Window trimming:** Removes contiguous blocks, capturing redundancies that single-letter removal misses.

## 3. Part 2 — Isomorphism

### Task 2(a): Strict Isomorphism

In [ ]:
import random

print('=== Task 2(a): Strict Isomorphism ===')

A = random_dfa(6, 2, seed=42)

# A DFA is isomorphic to itself
print(f'A ≅ A: {is_isomorphic(A, A)}')

# Relabel states: apply a permutation
Q = sorted(A[0])
perm = {Q[i]: Q[(i + 2) % len(Q)] for i in range(len(Q))}
Q2, Sigma, tau, q_s, F = A
B_tau = {(perm[q], a): perm[tau[(q, a)]] for q in Q2 for a in Sigma}
B = ({perm[q] for q in Q2}, Sigma, B_tau, perm[q_s], {perm[q] for q in F})
print(f'A ≅ relabelled(A): {is_isomorphic(A, B)}')

# Different DFA
C = random_dfa(6, 2, seed=99)
print(f'A ≅ C (different): {is_isomorphic(A, C)}')

# Different size
D = random_dfa(7, 2, seed=42)
print(f'A ≅ D (different size): {is_isomorphic(A, D)}')

### Task 2(b): Weak Isomorphism

In [ ]:
print('=== Task 2(b): Weak Isomorphism ===')

A = random_dfa(6, 2, seed=42)

# Relabel alphabet: swap 0 ↔ 1
Q_a, Sigma_a, tau_a, qs_a, F_a = A
alpha_perm = {0: 1, 1: 0}
E_tau = {(q, alpha_perm[a]): tau_a[(q, a)] for q in Q_a for a in Sigma_a}
E = (Q_a, Sigma_a, E_tau, qs_a, F_a)
print(f'A ~w alphabet-swapped(A): {is_weakly_isomorphic(A, E)}')

# Relabel both states and alphabet
perm = {Q[i]: Q[(i + 1) % len(Q)] for i in range(len(Q))}
F_tau = {(perm[q], alpha_perm[a]): perm[tau_a[(q, a)]] for q in Q_a for a in Sigma_a}
F_dfa = ({perm[q] for q in Q_a}, Sigma_a, F_tau, perm[qs_a], {perm[q] for q in F_a})
print(f'A ~w state+alpha relabelled(A): {is_weakly_isomorphic(A, F_dfa)}')

# Strict iso implies weak iso
print(f'Strict => Weak: is_isomorphic(A,A)={is_isomorphic(A,A)}, is_weakly_isomorphic(A,A)={is_weakly_isomorphic(A,A)}')

### Task 2(c): Semi-Isomorphism

In [ ]:
print('=== Task 2(c): Semi-Isomorphism ===')

A = random_dfa(6, 2, seed=42)
Q_a, Sigma_a, tau_a, qs_a, F_a = A

# Same transitions, different start state
B = (Q_a, Sigma_a, tau_a, (qs_a + 1) % len(Q_a), F_a)
print(f'Same transitions, different start: semi-iso = {is_semi_isomorphic(A, B)}')

# Same transitions, different final states
C = (Q_a, Sigma_a, tau_a, qs_a, Q_a - F_a)
print(f'Same transitions, different finals: semi-iso = {is_semi_isomorphic(A, C)}')

# Hierarchy: strict => weak => semi
print(f'\nHierarchy check on identical DFAs:')
print(f'  Strict: {is_isomorphic(A, A)}')
print(f'  Weak:   {is_weakly_isomorphic(A, A)}')
print(f'  Semi:   {is_semi_isomorphic(A, A)}')

### Isomorphism Performance

In [ ]:
print('Performance scaling (strict iso on identical DFAs):')
for n in [10, 50, 100, 500, 1000]:
    A = random_dfa(n, 2, seed=42)
    t0 = time.time()
    result = is_isomorphic(A, A)
    t1 = time.time()
    print(f'  n={n:5d}: {result}, time={t1-t0:.6f}s')

print('\nPerformance scaling (semi iso, same transitions different start):')
for n in [10, 50, 100, 200]:
    A = random_dfa(n, 2, seed=42)
    Q, S, t, qs, F = A
    B = (Q, S, t, (qs+1)%n, F)
    t0 = time.time()
    result = is_semi_isomorphic(A, B)
    t1 = time.time()
    print(f'  n={n:5d}: {result}, time={t1-t0:.6f}s')